# Seeded Run 1 Retrain v2 — 2-Class with AdamW + Lighter Regularization

**v1 issue:** weight_decay=5e-4 with Adam suppressed learning too aggressively (accuracy dropped ~10pp from original).

**v2 fixes:**
| Fix | v1 | v2 | Why |
|-----|-----|-----|-----|
| Optimizer | Adam | **AdamW** | Decoupled weight decay — more effective at lower strength |
| Weight decay | 5e-4 | **2e-4** | Lighter — preserve accuracy while controlling overfitting |
| LR | 2e-3 | 2e-3 | Keep |
| LR patience | 5 | 5 | Keep |

AdamW decouples weight decay from the adaptive gradient scaling, so 2e-4 in AdamW applies more consistent regularization than the same value in Adam.

In [1]:
import sys
sys.path.insert(0, r"C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files")

from pathlib import Path
import importlib

import pandas as pd
from IPython.display import display

import seeded_runs_common as seeded_runs_common

seeded_runs_common = importlib.reload(seeded_runs_common)

CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
expected_2class_odd_even_map = seeded_runs_common.expected_2class_odd_even_map
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

RUN_LABEL = "seeded_run_1_retrain_v2"
TASK_KEY = "2class"
MEM_DISTRIBUTION_FAMILY = "lognormal"
MASTER_SEEDS = [101, 202, 210, 340, 440]

RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
RESULT_STEM = RUN_DIR / f"{RUN_LABEL}_checkpoint_summary"
PAIR_STEM = RUN_DIR / f"{RUN_LABEL}_pair_summary"
PAIRED_ACC_CSV = RUN_DIR / f"{RUN_LABEL}_paired_accuracy.csv"

TASK_DEF = TASKS[TASK_KEY]
EXPECTED_LABEL_MAP = expected_2class_odd_even_map()

print(f"Device: {DEVICE}")
print(f"Run directory: {RUN_DIR}")
print(f"Master seeds: {MASTER_SEEDS}")
print(f"Task name: {TASK_DEF['task_name']}")

Device: cuda
Run directory: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1_retrain_v2\2class_lognormal
Master seeds: [101, 202, 210, 340, 440]
Task name: binary_parity


In [2]:
assert TASK_KEY == "2class"
assert TASK_DEF["nb_outputs"] == 2
assert TASK_DEF["task_name"] == "binary_parity"

preview = build_seeded_pair(
    master_seed=MASTER_SEEDS[0],
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
preview_meta = preview["metadata"]
assert preview["hom_prms"]["nb_outputs"] == 2
assert preview["hetero_prms"]["nb_outputs"] == 2
assert preview_meta["linear_sync_verified"]
assert preview_meta["hom_hidden_tau_unique"] == 1
assert preview_meta["hetero_hidden_tau_unique"] > 1

sampling_rows = build_sampling_preview_rows(
    MASTER_SEEDS, task_key=TASK_KEY, mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
sampling_df = pd.DataFrame(sampling_rows).sort_values("pair_seed").reset_index(drop=True)
assert not sampling_df["sample_matches_previous"].any()

display(sampling_df)
print("All verifications passed.")

,pair_seed,task_key,task_name,mem_distribution_family,linear_sync_verified,hetero_hidden_tau_unique,hom_hidden_tau_unique,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,sample_matches_previous
0,101,2class,binary_parity,lognormal,True,30,1,24.687252,3.410472,99.749887,37.499115,30.195999,False
1,202,2class,binary_parity,lognormal,True,28,1,29.308363,6.953492,99.749887,39.884979,30.480470,False
2,210,2class,binary_parity,lognormal,True,31,1,22.405558,3.855315,99.749887,34.397371,29.713145,False
3,340,2class,binary_parity,lognormal,True,31,1,23.273864,4.656051,99.749887,35.008043,30.389582,False
4,440,2class,binary_parity,lognormal,True,28,1,28.874491,5.547958,99.749887,41.448743,31.915950,False


All verifications passed.


In [3]:
train_cache, test_cache = load_default_caches()

Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5


In [4]:
# ── 2-Class Retrain v2: AdamW + Lighter Regularization ─────────────
#   1. optimizer Adam→AdamW: decoupled weight decay
#   2. weight_decay 5e-4→2e-4: lighter, preserve accuracy
#   3. batch_size 256, lr 2e-3, LR scheduler patience=5

import torch
import importlib

seeded_runs_common = importlib.reload(seeded_runs_common)

CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_seeded_pair = seeded_runs_common.build_seeded_pair
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

TASK_DEF = TASKS[TASK_KEY]

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

seeded_runs_common.BASE_PRMS["batch_size"] = 256          # OOM at 512
seeded_runs_common.BASE_PRMS["compile_model"] = False
seeded_runs_common.BASE_PRMS["optimizer"] = "adamw"       # AdamW instead of Adam
seeded_runs_common.BASE_PRMS["weight_decay"] = 2e-4       # was 5e-4: lighter
seeded_runs_common.BASE_PRMS["lr"] = 2e-3
seeded_runs_common.BASE_PRMS["lr_ab"] = 2e-3
seeded_runs_common.BASE_PRMS["lr_patience"] = 5
seeded_runs_common.BASE_PRMS["lr_factor"] = 0.5
seeded_runs_common.BASE_PRMS["lr_min"] = 5e-5

print("v2 settings applied:")
print(f"  optimizer      = AdamW")
print(f"  weight_decay   = {seeded_runs_common.BASE_PRMS['weight_decay']}  (was 5e-4)")
print(f"  batch_size     = 256")
print(f"  lr / lr_ab     = {seeded_runs_common.BASE_PRMS['lr']:.0e}")
print(f"  LR scheduler   = ReduceLROnPlateau(patience=5)")


v2 settings applied:
  optimizer      = AdamW
  weight_decay   = 0.0002  (was 5e-4)
  batch_size     = 256
  lr / lr_ab     = 2e-03
  LR scheduler   = ReduceLROnPlateau(patience=5)


In [5]:
CHECKPOINT_KEY_FIELDS = ["pair_seed", "pair_role"]
PAIR_KEY_FIELDS = ["pair_seed"]

def show_run_status():
    checkpoint_rows = read_manifest_rows(RESULT_STEM)
    pair_rows = read_manifest_rows(PAIR_STEM)
    checkpoint_df = pd.DataFrame(checkpoint_rows)
    pair_df = pd.DataFrame(pair_rows)
    if not checkpoint_df.empty:
        checkpoint_df = checkpoint_df.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    if not pair_df.empty:
        pair_df = pair_df.sort_values("pair_seed").reset_index(drop=True)
    paired_acc_df = pd.DataFrame()
    if not checkpoint_df.empty:
        paired_acc_df = (
            checkpoint_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
            .reset_index().sort_values("pair_seed").reset_index(drop=True)
        )
        if {"heterogeneous_sampled", "homogeneous_anchor"}.issubset(paired_acc_df.columns):
            paired_acc_df["hetero_minus_hom"] = (
                paired_acc_df["heterogeneous_sampled"] - paired_acc_df["homogeneous_anchor"]
            )
            paired_acc_df.to_csv(PAIRED_ACC_CSV, index=False)
    return pair_df, checkpoint_df, paired_acc_df

def train_one_seed(master_seed):
    run_rows, pair_meta = run_pair_training(
        master_seed=master_seed, train_cache=train_cache, test_cache=test_cache,
        task_key=TASK_KEY, mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
        run_label=RUN_LABEL, skip_existing=True,
    )
    checkpoint_rows = upsert_rows(read_manifest_rows(RESULT_STEM), run_rows, CHECKPOINT_KEY_FIELDS)
    pair_rows = upsert_rows(read_manifest_rows(PAIR_STEM), [build_pair_summary_row(pair_meta)], PAIR_KEY_FIELDS)
    write_manifest_rows(checkpoint_rows, RESULT_STEM)
    write_manifest_rows(pair_rows, PAIR_STEM)
    pair_df, checkpoint_df, paired_acc_df = show_run_status()
    seed_df = checkpoint_df[checkpoint_df["pair_seed"] == master_seed].reset_index(drop=True)
    return seed_df, pair_df, checkpoint_df, paired_acc_df

print("Helpers ready. Chance: 50.0%")

Helpers ready. Chance: 50.0%


In [6]:
# Seed 101
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(101)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 101].reset_index(drop=True))


Seed 101 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.554  test_acc=0.596  (3.2 min)
  epoch=002  train_acc=0.632  test_acc=0.645  (3.6 min)
  epoch=003  train_acc=0.700  test_acc=0.708  (3.5 min)
  epoch=004  train_acc=0.726  test_acc=0.714  (3.4 min)
  epoch=005  train_acc=0.743  test_acc=0.703  (3.5 min)
  epoch=006  train_acc=0.760  test_acc=0.674  (3.9 min)
  epoch=007  train_acc=0.772  test_acc=0.631  (3.4 min)
  epoch=008  train_acc=0.786  test_acc=0.706  (3.4 min)
  epoch=009  train_acc=0.799  test_acc=0.715  (3.4 min)
  epoch=010  train_acc=0.800  test_acc=0.722  (3.3 min)
  epoch=011  train_acc=0.808  test_acc=0.745  (3.4 min)
  epoch=012  train_acc=0.815  test_acc=0.714  (3.2 min)
  epoch=013  train_acc=0.823  test_acc=0.725  (3.8 min)
  epoch=014  train_acc=0.832  test_acc=0.773  (3.6 min)
  epoch=015  train_acc=0.832  test_acc=0.734  (3.1 min)
  epoch=016  train_acc=0.843  test_acc=0.745  (3.3 min)
  epoch=017  train_acc=0.849  test_acc=0.761  (3.2 min)
  epoch=018  train_acc=0.851  test_acc=0.766  (3

,pair_seed,pair_role,final_test_acc,final_test_loss,elapsed_s
0,101,heterogeneous_sampled,0.816696,0.405430,4191.277313
1,101,homogeneous_anchor,0.763251,0.522708,5028.456070


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,101,2class,binary_parity,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,30,1,True,6


In [7]:
# Seed 202
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(202)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 202].reset_index(drop=True))


Seed 202 :: training homogeneous_anchor...


c:\Users\Priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  epoch=001  train_acc=0.539  test_acc=0.650  (2.7 min)
  epoch=002  train_acc=0.658  test_acc=0.696  (2.8 min)
  epoch=003  train_acc=0.688  test_acc=0.703  (2.8 min)
  epoch=004  train_acc=0.707  test_acc=0.722  (2.9 min)
  epoch=005  train_acc=0.715  test_acc=0.623  (2.8 min)
  epoch=006  train_acc=0.736  test_acc=0.671  (2.6 min)


KeyboardInterrupt: 

In [ ]:
# Seed 210
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(210)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 210].reset_index(drop=True))

In [ ]:
# Seed 340
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(340)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 340].reset_index(drop=True))

In [ ]:
# Seed 440
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(440)
display(seed_df[["pair_seed", "pair_role", "final_test_acc", "final_test_loss", "elapsed_s"]])
display(pair_df[pair_df["pair_seed"] == 440].reset_index(drop=True))

In [ ]:
# Summary + Stats
pair_df, checkpoint_df, paired_acc_df = show_run_status()

if not paired_acc_df.empty:
    display(paired_acc_df)
    print(f"Saved: {PAIRED_ACC_CSV}")
    
    import numpy as np
    from scipy import stats
    
    diffs = paired_acc_df["hetero_minus_hom"].values
    homo = paired_acc_df["homogeneous_anchor"].values
    hetero = paired_acc_df["heterogeneous_sampled"].values
    n = len(diffs)
    mean_diff = np.mean(diffs)
    
    if n >= 2:
        t_stat, p_ttest = stats.ttest_rel(hetero, homo, alternative="greater")
        w_stat, p_wilcoxon = stats.wilcoxon(hetero, homo, alternative="greater")
        cohens_d = mean_diff / np.std(diffs, ddof=1)
        
        print(f"\nMean Δ: {mean_diff*100:+.2f} pp | Cohen's d: {cohens_d:.3f}")
        print(f"t-test p: {p_ttest:.4f} | Wilcoxon p: {p_wilcoxon:.4f}")
        print(f"Seeds hetero>homo: {int(np.sum(diffs>0))}/{n}")
        
        # Compare to v1
        print(f"\nvs v1 (Adam, wd=5e-4): mean Δ=+2.41pp, Cohen's d=0.82, p=0.07")
        print(f"vs original: mean Δ=+2.69pp (overfit)")